# CrudeWatch — Clean WTI Dataset

All cleaning and spread-construction logic lives in the `crudewatch` package (`src/crudewatch/`). This notebook just drives it: load the raw feed, build every dataframe, and plot a few contracts with the shared black/green theme.


In [1]:
import pandas as pd

from crudewatch.infra import load_raw
from crudewatch.data_preparation import build_all, summarize
from crudewatch.plots import line_figure

## Load the raw feed


In [2]:
RAW = "/Users/I0551731/Desktop/CrudeWatch/data/raw_files.xlsx"
df = load_raw(RAW)
df.head()

,ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,date
0,2010-06-06T00:00:00.000000000Z,35,1,182059,74.63,74.67,74.38,74.52,9,CLX0,2010-06-06
1,2010-06-06T00:00:00.000000000Z,35,1,182065,78.59,78.59,78.50,78.50,12,CLM1,2010-06-06
2,2010-06-06T00:00:00.000000000Z,35,1,182056,71.83,71.90,71.25,71.70,599,CLQ0,2010-06-06
3,2010-06-06T00:00:00.000000000Z,35,1,182010,75.39,75.44,75.11,75.20,96,CLZ0,2010-06-06
4,2010-06-06T00:00:00.000000000Z,35,1,182058,74.20,74.21,73.43,73.64,81,CLV0,2010-06-06


## Build every dataframe

`build_all` returns the three exchange-listed families (`outrights`, `calendars`, `cracks`) plus the synthetic structures built from outright legs (`quarterly` = M vs M+3, `semestral` = M vs M+6, `yearly` = M vs M+12, and same-month consecutive-year `flies`).


In [3]:
frames = build_all(df)

outrights = frames["outrights"]
calendars = frames["calendars"]   # exchange-listed calendar spreads
cracks    = frames["cracks"]
brent_wti = frames["brent_wti"]   # Brent - WTI premium (BZ vs CL)
quarterly = frames["quarterly"]   # synthetic, M vs M+3
semestral = frames["semestral"]   # synthetic, M vs M+6
yearly    = frames["yearly"]      # synthetic, M vs M+12 (e.g. Dec/Dec)
flies     = frames["flies"]       # synthetic, same-month butterflies

summarize(frames)


Summary
----------------------------------------------------------------
outrights  rows:    60302 | contracts:    246
calendars  rows:   358678 | contracts:   3219
cracks     rows:    42418 | contracts:    383
brent_wti  rows:    41004 | contracts:    410
quarterly  rows:    52291 | contracts:    221
semestral  rows:    45970 | contracts:    223
yearly     rows:    30089 | contracts:    222
flies      rows:    10770 | contracts:    186


In [4]:
yearly.head()

,date,near_contract,near_close,near_month,near_month_code,near_year,far_contract,far_close,close,contract,structure,gap_months,n_legs
0,2010-06-06,CLZ2010,75.20,12,Z,2010,CLZ2011,79.70,-4.50,CLZ2010-CLZ2011,yearly,12,2
1,2010-06-07,CLN2010,70.99,7,N,2010,CLN2011,79.21,-8.22,CLN2010-CLN2011,yearly,12,2
2,2010-06-07,CLQ2010,72.27,8,Q,2010,CLQ2011,79.92,-7.65,CLQ2010-CLQ2011,yearly,12,2
3,2010-06-07,CLU2010,73.21,9,U,2010,CLU2011,80.04,-6.83,CLU2010-CLU2011,yearly,12,2
4,2010-06-07,CLV2010,73.97,10,V,2010,CLV2011,80.24,-6.27,CLV2010-CLV2011,yearly,12,2


## WTI outright — CLZ2022


In [5]:
series = outrights.query("contract == 'CLZ2022'").sort_values("date")
fig = line_figure(
    series,
    title="WTI Outright \u2014 CLZ2022",
    y_title="Close ($/bbl)",
    fill_to_zero=False
)
fig.show()

## Calendar spread — Dec/Dec (CLZ2022-CLZ2023)

Synthetic yearly spread, front minus back (positive = backwardation).


In [6]:
series = yearly.query("contract == 'CLZ2022-CLZ2023'").sort_values("date")
fig = line_figure(
    series,
    title="WTI Calendar \u2014 Dec22/Dec23 (CLZ2022-CLZ2023)",
    y_title="Spread ($/bbl)",
    fill_to_zero=False,
)
fig.show()

## Butterfly — Dec22/Dec23/Dec24

Synthetic fly: `P(front) - 2*P(mid) + P(back)`.


In [7]:
series = flies.query("contract == 'CLZ2022-CLZ2023-CLZ2024'").sort_values("date")
fig = line_figure(
    series,
    title="WTI Butterfly \u2014 Dec22/Dec23/Dec24",
    y_title="Fly ($/bbl)",
    fill_to_zero=False,
)
fig.show()

## Crack spreads


In [8]:
series = cracks.query("contract == 'CL:C1 RB-CL Z2022'").sort_values("date")
fig = line_figure(
    series,
    title="RBOB\u2013WTI Crack Spread \u2014 CL:C1 RB-CL Z2022",
    y_title="Crack ($/bbl)",
    fill_to_zero=False,
)
fig.show()

In [9]:
series = cracks.query("contract == 'CL:C1 HO-CL Z2022'").sort_values("date")
fig = line_figure(
    series,
    title="HO\u2013WTI Crack Spread \u2014 CL:C1 HO-CL Z2022",
    y_title="Crack ($/bbl)",
    fill_to_zero=False,
)
fig.show()

## Brent–WTI spread — Dec 2022 (CLZ2022-BZZ2022)

The Brent premium over WTI (`Brent − WTI`, positive = Brent richer).

In [10]:
series = brent_wti.query("contract == 'CLZ2022-BZZ2022'").sort_values("date")
fig = line_figure(
    series,
    title="Brent\u2013WTI Spread \u2014 Dec 2022 (CLZ2022-BZZ2022)",
    y_title="Brent \u2212 WTI ($/bbl)",
    fill_to_zero=False,
)
fig.show()